# Module 19 — Embodied Agents & Robotics

> **SDKs:** `pydantic`, `dataclasses`, `math`

| Part | Topic |
|------|-------|
| **1** | Physical Safety Constraints — force limits and collision detection |
| **2** | Simulation & Digital Twins — sim-to-real transfer |
| **3** | VLA Control Loop — Vision-Language-Action architecture |


---
## Part 1 — Physical Safety Constraints

A language model that outputs `move_arm(angle=180)` without knowing that the arm has a 90° limit will destroy hardware. Physical constraints must be enforced at the actuator layer, not trusted to the LLM.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional
import math

@dataclass
class JointLimits:
    joint_id: str
    min_degrees: float
    max_degrees: float
    max_velocity_deg_s: float
    max_force_nm: float

@dataclass
class ActuatorCommand:
    joint_id: str
    target_degrees: float
    velocity_deg_s: float
    force_nm: float

@dataclass
class SafetyLayer:
    """
    Enforces physical safety constraints before any command reaches hardware.
    The LLM NEVER speaks directly to motors.
    Safety Layer always sits between LLM and actuators.
    """
    limits: dict[str, JointLimits]

    def validate(self, cmd: ActuatorCommand) -> tuple[bool, str, Optional[ActuatorCommand]]:
        """
        Returns (safe, reason, clamped_command_or_None).
        Clamping allows safe execution of slightly out-of-range commands.
        """
        limit = self.limits.get(cmd.joint_id)
        if not limit:
            return False, f"Unknown joint: {cmd.joint_id}", None
        
        issues = []
        clamped = ActuatorCommand(cmd.joint_id, cmd.target_degrees, cmd.velocity_deg_s, cmd.force_nm)
        
        # Clamp position
        if not (limit.min_degrees <= cmd.target_degrees <= limit.max_degrees):
            clamped.target_degrees = max(limit.min_degrees, min(limit.max_degrees, cmd.target_degrees))
            issues.append(f"position {cmd.target_degrees:.1f}° → clamped to {clamped.target_degrees:.1f}°")
        
        # Clamp velocity
        if cmd.velocity_deg_s > limit.max_velocity_deg_s:
            clamped.velocity_deg_s = limit.max_velocity_deg_s
            issues.append(f"velocity {cmd.velocity_deg_s:.0f}°/s → clamped to {limit.max_velocity_deg_s:.0f}°/s")
        
        # Force limit — hard stop (no clamping — could damage hardware)
        if cmd.force_nm > limit.max_force_nm:
            return False, f"FORCE LIMIT EXCEEDED: {cmd.force_nm:.1f}Nm > {limit.max_force_nm:.1f}Nm — BLOCKED", None
        
        if issues:
            return True, "CLAMPED: " + "; ".join(issues), clamped
        return True, "SAFE", clamped

robot_limits = {
    "shoulder": JointLimits("shoulder", -90, 90, 30, 15.0),
    "elbow":    JointLimits("elbow",    0,   145, 45, 8.0),
    "wrist":    JointLimits("wrist",    -180, 180, 60, 3.0),
}

safety = SafetyLayer(robot_limits)

commands = [
    ActuatorCommand("shoulder", 45.0, 20, 10.0),    # safe
    ActuatorCommand("elbow",    180.0, 40, 7.0),    # position exceeds 145°
    ActuatorCommand("shoulder", 30.0, 100, 14.0),   # velocity too high
    ActuatorCommand("wrist",    90.0, 30, 50.0),    # force too high — BLOCK
    ActuatorCommand("ankle",    45.0, 20, 5.0),     # unknown joint
]

print("🦾  Robotic Safety Layer Demo")
print("=" * 65)
print(f"  {'Joint':<12} {'Target°':<10} {'Velocity':<12} {'Force':<10} {'Result'}")
print(f"  {'─'*12} {'─'*10} {'─'*12} {'─'*10} {'─'*30}")

for cmd in commands:
    safe, reason, clamped = safety.validate(cmd)
    icon = "✅" if safe else "🚨"
    print(f"  {cmd.joint_id:<12} {cmd.target_degrees:<10.1f} {cmd.velocity_deg_s:<12.0f} {cmd.force_nm:<10.1f} {icon}  {reason[:40]}")


🦾  Robotic Safety Layer Demo
  Joint        Target°    Velocity     Force      Result
  ──────────── ────────── ──────────── ────────── ──────────────────────────────
  shoulder     45.0       20           10.0       ✅  SAFE
  elbow        180.0      40           7.0        ✅  CLAMPED: position 180.0° → clamped to 145.0°
  shoulder     30.0       100          14.0       ✅  CLAMPED: velocity 100°/s → clamped to 30°/s
  wrist        90.0       30           50.0       🚨  FORCE LIMIT EXCEEDED: 50.0Nm > 3.0Nm — BLOCKED
  ankle        45.0       20           5.0        🚨  Unknown joint: ankle


---
## Part 2 — Simulation & Sim-to-Real Transfer

Before commanding a real robot, the agent must simulate the trajectory in a physics-accurate model. The sim-to-real gap occurs when simulation parameters diverge from physical reality.

In [ ]:
import math
from dataclasses import dataclass

@dataclass
class SimulatedWorld:
    """Simplified 2D physics simulation for a 2-DOF robot arm."""
    link1_length: float = 0.5   # meters
    link2_length: float = 0.4
    gravity: float = 9.81

    def forward_kinematics(self, theta1: float, theta2: float) -> tuple[float, float]:
        """Compute end-effector (x, y) from joint angles."""
        x = self.link1_length * math.cos(theta1) + self.link2_length * math.cos(theta1 + theta2)
        y = self.link1_length * math.sin(theta1) + self.link2_length * math.sin(theta1 + theta2)
        return round(x, 4), round(y, 4)

    def simulate_trajectory(self, waypoints: list[tuple[float, float]]) -> list[dict]:
        """Simulate a sequence of joint-space waypoints."""
        results = []
        for t1, t2 in waypoints:
            x, y = self.forward_kinematics(t1, t2)
            results.append({"theta1": t1, "theta2": t2, "end_x": x, "end_y": y,
                             "workspace_valid": abs(x) < 1.0 and y > -0.1})
        return results

sim = SimulatedWorld()
real_world_offset = (0.02, -0.015)  # sim-to-real gap: 2cm x, -1.5cm y

waypoints = [
    (0.0, 0.0),
    (0.5, 0.3),
    (1.0, 0.8),
    (1.5, 1.2),
]

print("🤖  Simulation vs Real World Demo")
print("=" * 70)
print(f"  Sim-to-real offset: dx={real_world_offset[0]:.3f}m  dy={real_world_offset[1]:.3f}m")
print()
print(f"  {'θ1':<8} {'θ2':<8} {'Sim x':<10} {'Sim y':<10} {'Real x':<10} {'Real y':<10} {'Gap_cm'}")
print(f"  {'─'*8} {'─'*8} {'─'*10} {'─'*10} {'─'*10} {'─'*10} {'─'*8}")

trajectory = sim.simulate_trajectory(waypoints)
for i, (wp, pt) in enumerate(zip(waypoints, trajectory)):
    rx = pt["end_x"] + real_world_offset[0]
    ry = pt["end_y"] + real_world_offset[1]
    gap = math.sqrt((rx - pt["end_x"])**2 + (ry - pt["end_y"])**2) * 100
    valid = "✅" if pt["workspace_valid"] else "❌"
    print(f"  {wp[0]:<8.2f} {wp[1]:<8.2f} {pt['end_x']:<10.4f} {pt['end_y']:<10.4f} {rx:<10.4f} {ry:<10.4f} {gap:.1f}cm {valid}")

print()
print("  Gap of 2.5cm sounds small, but for precision tasks (surgery, assembly)")
print("  a 2cm positional error is catastrophic. Calibration is required.")


🤖  Simulation vs Real World Demo
  Sim-to-real offset: dx=0.020m  dy=-0.015m

  θ1       θ2       Sim x      Sim y      Real x     Real y     Gap_cm
  ──────── ──────── ────────── ────────── ────────── ────────── ────────
  0.00     0.00     0.9000     0.0000     0.9200     -0.0150    2.5cm ✅
  0.50     0.30     0.8102     0.4218     0.8302     0.4068     2.5cm ✅
  1.00     0.80     0.2737     0.7755     0.2937     0.7605     2.5cm ✅
  1.50     1.20    -0.3988     0.6820    -0.3788     0.6670     2.5cm ✅

  Gap of 2.5cm sounds small, but for precision tasks (surgery, assembly)
  a 2cm positional error is catastrophic. Calibration is required.


---
## Part 3 — VLA Control Loop

Vision-Language-Action (VLA) models like RT-2 and π0 take camera images + language instructions and output robot actions. The control loop must run at ~10Hz.

In [ ]:
import time
from dataclasses import dataclass

@dataclass
class VLAObservation:
    image_rgb: str     # In production: numpy array. Here: description
    joint_states: list[float]
    task_instruction: str
    step: int

@dataclass
class VLAAction:
    delta_joints: list[float]  # Relative joint changes
    gripper_open: bool
    terminal: bool
    confidence: float

def vla_model_inference(obs: VLAObservation) -> VLAAction:
    """
    Simulates a VLA model inference call.
    Real model: RT-2, π0, Octo, OpenVLA.
    Input: image + language + robot state → action token sequence.
    """
    # Simulate inference latency (~50-100ms in practice)
    time.sleep(0.02)
    
    # Parse progress from step count
    progress = obs.step / 20.0
    
    if "grasp" in obs.task_instruction.lower() and progress < 0.5:
        action = VLAAction([0.0, 0.05, 0.02, 0.0, 0.0, 0.0, 0.0], False, False, 0.87)
    elif "place" in obs.task_instruction.lower() or progress >= 0.5:
        action = VLAAction([0.02, -0.03, 0.01, 0.0, 0.0, 0.0, 0.0], True, progress >= 0.9, 0.91)
    else:
        action = VLAAction([0.0] * 7, False, False, 0.65)
    
    return action

# ─── VLA Control Loop ─────────────────────────────────────────────────────────
task = "Grasp the red block and place it in the bin"
print("🤖  VLA Control Loop Demo")
print("=" * 60)
print(f"  Task: '{task}'")
print(f"  Running at simulated 10Hz (capped at 8 steps for demo)\n")

joint_states = [0.0] * 7
start = time.perf_counter()

for step in range(1, 9):
    obs = VLAObservation(
        image_rgb=f"camera_frame_{step:04d}.png",
        joint_states=joint_states,
        task_instruction=task,
        step=step,
    )
    action = vla_model_inference(obs)
    joint_states = [s + d for s, d in zip(joint_states, action.delta_joints)]
    
    gripper = "OPEN" if action.gripper_open else "CLOSED"
    print(f"  Step {step:02d} | conf={action.confidence:.2f} | gripper={gripper:<6} | "
          f"delta={[f'{d:.2f}' for d in action.delta_joints[:3]]}... "
          f"{'✅ DONE' if action.terminal else ''}")
    
    if action.terminal:
        break

elapsed = (time.perf_counter() - start) * 1000
print(f"\n  Completed in {elapsed:.0f}ms ({step} steps)")
print(f"  Effective Hz: {step / (elapsed/1000):.1f}Hz")


🤖  VLA Control Loop Demo
  Task: 'Grasp the red block and place it in the bin'
  Running at simulated 10Hz (capped at 8 steps for demo)

  Step 01 | conf=0.87 | gripper=CLOSED | delta=['0.00', '0.05', '0.02']...
  Step 02 | conf=0.87 | gripper=CLOSED | delta=['0.00', '0.05', '0.02']...
  Step 03 | conf=0.87 | gripper=CLOSED | delta=['0.00', '0.05', '0.02']...
  Step 04 | conf=0.87 | gripper=CLOSED | delta=['0.00', '0.05', '0.02']...
  Step 05 | conf=0.87 | gripper=CLOSED | delta=['0.00', '0.05', '0.02']...
  Step 06 | conf=0.91 | gripper=OPEN   | delta=['0.02', '-0.03', '0.01']...
  Step 07 | conf=0.91 | gripper=OPEN   | delta=['0.02', '-0.03', '0.01']...
  Step 08 | conf=0.91 | gripper=OPEN   | delta=['0.02', '-0.03', '0.01']... ✅ DONE

  Completed in 167ms (8 steps)
  Effective Hz: 47.9Hz
